In [ ]:
%load_ext autoreload
%autoreload 1

%aimport highlighting_precision
%aimport gradcam_APs
%aimport gradcam
%aimport vanilla_vgg
%aimport dataHelper
%aimport external

import warnings
from torch.serialization import SourceChangeWarning
warnings.filterwarnings("ignore", category=SourceChangeWarning)

%matplotlib inline
!nvidia-smi

In [ ]:
# Determine Device from Above
import torch
torch.cuda.set_device(1)

In [ ]:
# imports
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ext = external.ExternalValidationContext(
    environment = 'emory',
    external_validation_path=Path('/redacted/path/to/emory'),
    data_path=Path('/redacted/path/to/data'),
    models=['iaia-bl', 'protopnet',  'vgg16'],
    margin_name_labels = [
        (0, 'circ_prob', 'circumscribed'),
        (1, 'indst_prob', 'indistinct'),
        (2, 'spic_prob', 'spiculated')
    ],
    malignancy_name_labels = [
        (0, 'mal_neg_prob', 'not_malignant'),
        (1, 'mal_pos_prob', 'malignant')
    ],
    subgroupings  = [
        ('all', 'all'), 
        ('manufacturer', 'ge'), ('manufacturer', 'hologic'),
        ('age_3_bucket', '<50'), ('age_3_bucket','50-69'), ('age_3_bucket', '>70'),
        ('age_2_bucket', '<=57'), ('age_2_bucket','>57'),
        ('race', 'Black or African American'), ('race', 'Caucasian/White'), ('race', 'Other/Unknown'),
        ('ethnicity', 'Hispanic or Latino'), ('ethnicity', 'Not Hispanic or Latino')
    ]
)

ext

# Load Emory Metadata

In [ ]:
basepath = Path('/redacted/path/to/tables')
KEY_COLUMNS_MAG = ['index', 'empi_anon', 'acc_anon', 'side', 'numfind', 'desc', 'asses']
KEY_COLUMNS_MET = ['Unnamed: 0', 'empi_anon', 'acc_anon', 'ImageLateralityFinal', 'ViewPosition', 'FinalImageType', 'AcquisitionTime']
CONTEXT_COLUMNS = ['cohort_num', 'study_date_anon']

__birads_map = {
    'A': 0,
    'N': 1,
    'B': 2,   
    'P': 3,
    'S': 4,
    'M': 5,
    'K': 6,
    'X': 'X'
}
    # BIRADS 0: A – Additional evaluation
    # BIRADS 1: N – Negative
    # BIRADS 2: B - Benign
    # BIRADS 3: P – Probably benign
    # BIRADS 4: S – Suspicious
    # BIRADS 5: M- Highly suggestive of malignancy
    # BIRADS 6: K - Known biopsy proven

def translate_birads(hiti_asses):
    return __birads_map[hiti_asses]


def lead_with_columns(df, lead_columns):
    """
    Rearranges dataframe column order by pulling a few columns to the front but keeping the order for other columns
    """
    post_columns = [col for col in df.columns if col not in lead_columns]
    return df[lead_columns + post_columns]

def load_met(file_path=os.path.join(basepath, 'metadata_all_cohort_with_ROI_HITI.csv'), 
            scope='min', extra_cols=[], cohorts=None, nrows=None):
    
    force_string_cols = ['CollimatorShape', '0_ProcedureCodeSequence_CodeValue', 'DerivationDescription',
                        'CommentsOnRadiationDose', 'DetectorDescription', 'WindowCenter', 'WindowWidth']
    
    categorical_cols = ['AcquisitionDeviceProcessingCode', 'AcquisitionDeviceProcessingDescription',
                        'DetectorConfiguration', 'FieldOfViewShape', 'CollimatorShape',
                        'DetectorActiveShape', 'ExposureStatus', 'VOILUTFunction'
                        '0_IconImageSequence_PhotometricInterpretation']
    
    tuple_cols = ['FieldOfViewDimensions', 'DetectorActiveDimensions', 'DetectorElementPhysicalSize',
                'DetectorElementSpacing', 'WindowCenterWidthExplanation']
    
    date_cols = ['study_date_anon']
    
    forced_types = {col: 'string' for col in force_string_cols}
    forced_types.update({col: 'category' for col in categorical_cols})
    forced_types.update({col: 'string' for col in tuple_cols})
    
    if scope == 'min':
        usecols = (KEY_COLUMNS_MET + CONTEXT_COLUMNS + 
                    ['png_path', 'png_filename', 'num_roi', 'ROI_coords', 'match_level', 'spot_mag'] + extra_cols)
    
    elif scope == 'full':
        usecols = None
    
    else:
        return NotImplemented(f'{scope} is not a valid column scope')
    
    df_metadata = pd.read_csv(file_path, dtype=forced_types, parse_dates=date_cols, usecols=usecols, nrows=nrows)
    
    if cohorts:
        df_metadata = df_metadata.loc[df_metadata['cohort_num'].isin(cohorts)]

    return df_metadata.pipe(lead_with_columns, KEY_COLUMNS_MET + CONTEXT_COLUMNS)

def load_mag(file_path=basepath/'magview_all_cohorts_anon_HITI.csv', scope = 'min', extra_cols=[], cohorts=None):
    # columns that have inconsistent value types - default casting doesn't work properly
    inconsistent_columns = ['case', 'biopsite', 'bcomp', 'path7', 'path8', 'path9', 'path10', 'hgrade',
                            'tnmpt', 'tnmpn', 'tnmm', 'tnmdesc', 'stage', 'bdepth','focality',
                            'specinteg', 'specembed', 'her2', 'fish', 'extracap', 'methodevl',
                            'eic', 'first_3_zip']
    type_overrides = {col: 'string' for col in inconsistent_columns}

    categories = ['massshape', 'massmargin', 'massdens', 'calcfind', 'calcdistri', 'otherfind', 
                'implanfind', 'side', 'location', 'depth', 'distance', 'asses', 'recc', 'proccode',
                'vtype','tissueden', 'MARITAL_STATUS_DESC']
    type_overrides.update({col: 'category' for col in categories})

    date_cols = {'study_date_anon','sdate_anon','procdate_anon','pdate_anon'}
    
    if scope == 'min':
        usecols = (KEY_COLUMNS_MAG + CONTEXT_COLUMNS + ['ETHNIC_GROUP_DESC', 'ETHNICITY_DESC', 'age_at_study'] +
            ['massshape', 'massmargin', 'massdens', 'otherfind', 'path_group', 'path_severity'] + extra_cols)
        date_cols = date_cols.intersection(usecols)
        
    elif scope == 'full':
        usecols = None
        
    else:
        return NotImplemented(f'{scope} is not a valid column scope')
    
    df_mag =  pd.read_csv(file_path, header=0,usecols=usecols)
    
    df_mag['scr_or_diag'] = df_mag['desc'].apply(lambda desc: 'S' if 'screen' in desc.lower() else 'D').astype("category")
    df_mag['birads'] = df_mag['asses'].apply(translate_birads)

    if cohorts:
        df_mag = df_mag.loc[df_mag['cohort_num'].isin(cohorts)]

    return df_mag.pipe(lead_with_columns, KEY_COLUMNS_MAG + CONTEXT_COLUMNS)

# Emory Validation

## Activation Precision
These just setup the run and get preliminary results (n<<100). See the Export section below for n=100.

### GradCAM for VGG16

In [ ]:
gradcam_aps = ext.run_gradcams(num_bootstraps=3)
gradcam_aps.pipe(display)
gradcam_aps.groupby(['method']).mean().pipe(display)

### ProtoPNets

In [ ]:
protopnets_aps = ext.classwise_activation_precision_for_protopnets(num_bootstraps=3)
protopnets_aps.pipe(display)
protopnets_aps.groupby(['model']).mean().pipe(display)

# Model Predictions
## Margin Predictions

In [ ]:
margin_predictions = ext.margin_predictions_for_models(image_has_rois=True)
margin_predictions['image'] = margin_predictions['patient_id'].str.split('_', 0, expand=True)[0]
margin_predictions = margin_predictions.drop('patient_id', axis=1)
margin_predictions.to_csv(ext.external_validation_path/f'{ext.environment}_margin_predictions.csv', index=False)
margin_predictions.groupby('model_name').count()

In [ ]:
__prob_sums = margin_predictions[['circ_prob', 'indst_prob', 'spic_prob']].sum(axis=1).round(2).value_counts()
assert len(__prob_sums) == 1 and 1.0 in __prob_sums
__prob_sums

### Analyze Results

In [ ]:
margin_predictions = pd.read_csv(ext.external_validation_path/f'{ext.environment}_margin_predictions.csv')
margin_predictions.groupby('model_name').apply(lambda group: group.sample(2))

In [ ]:
ext.plot_margin_probs(margin_predictions)

# Data Augmentation
Needed for malignancy prediction

In [ ]:
label_file = Path('/redacted/path/to/dataset/index.csv')
df_met_4path = pd.read_csv(label_file)
print(df_met_4path.shape)
df_met_4path['image'] = df_met_4path['original_image'].str.split('.', 0, expand=True)[0]#remove .png
df_met_4path['acc_anon_scr'].unique().shape

In [ ]:
mag = load_mag()
mag.head()

In [ ]:
mag['empi_anon'].nunique()

In [ ]:
# Adding in columns for race and age
df_met_4path_wmag = df_met_4path.merge(mag, left_on=['acc_anon_scr', 'ImageLateralityFinal'], right_on=['acc_anon', 'side'])
df_met_4path_wmag.head()

In [ ]:
margin_predictions['image'][0], df_met_4path_wmag['image'][0]

In [ ]:
margin_pred_wlabels_df = margin_predictions.merge(df_met_4path, left_on='image', right_on='image')

assert margin_pred_wlabels_df.shape[0] == margin_pred_wlabels_df['image'].nunique() * 3
margin_pred_wlabels_df.head(2).pipe(display)
margin_pred_wlabels_df.shape[0]

In [ ]:
margin_pred_wlabels_df.columns

In [ ]:
mass_uid = (margin_pred_wlabels_df['image'].astype(str) + margin_pred_wlabels_df['side'])
mass_uid.nunique(), margin_pred_wlabels_df['acc_anon_scr'].shape

In [ ]:
score_label_df = margin_pred_wlabels_df.merge(mag, left_on=['empi_anon', 'acc_anon_scr', 'side'], right_on=['empi_anon', 'acc_anon', 'side'])
display(score_label_df.shape)
score_label_df.head()

In [ ]:
margin_pred_wlabels_df['Manufacturer'].unique()

In [ ]:
score_label_df['mass_uid'] = score_label_df['image'] + score_label_df['side']
score_label_df['image_id'] = score_label_df['image']
score_label_df['patient_id'] = score_label_df['empi_anon']

def clean_manufacturer(manufacturer):
    if manufacturer == 'HOLOGIC, Inc.':
        return 'hologic'
    else:
        raise ValueError(manufacturer)

def clean_race(row):
    race, ethnicity = row['ETHNICITY_DESC'], row['ETHNIC_GROUP_DESC']
    # TODO - check these labels
    if race == 'African American  or Black':
        return 'Black or African American'
    elif race == 'Caucasian or White':
        return 'Caucasian/White'
    elif race in ('Unknown, Unavailable or Unreported', 'Asian', 'Native Hawaiian or Other Pacific Islander', 'American Indian or Alaskan Native'):
        return 'Other/Unknown'
    else:
        raise ValueError(race)

score_label_df['race'] = score_label_df[['ETHNICITY_DESC', 'ETHNIC_GROUP_DESC']].apply(clean_race, axis=1)
score_label_df['ethnicity'] = score_label_df['ETHNIC_GROUP_DESC']

score_label_df['age'] = score_label_df['age_at_study']
score_label_df['age_3_bucket'] = score_label_df['age_at_study'].apply(external.age_3_bucket)
score_label_df['age_2_bucket'] = score_label_df['age_at_study'].apply(external.age_2_bucket)

score_label_df['manufacturer_orig'] = score_label_df['Manufacturer']
score_label_df['manufacturer'] = score_label_df['manufacturer_orig'].apply(clean_manufacturer)

score_label_df[['ETHNICITY_DESC', 'race', 'age', 'age_2_bucket', 'age_3_bucket', 'manufacturer_orig', 'manufacturer']].head(3)

In [ ]:
path_severity_map = {
    0: "Invasive Cancer",
    1: "Non-Invasive Cancer",
    2: "High-Risk Lesion",
    3: "Borderline Lesion",
    4: "Benign Finding"
}

score_label_df['malignancy_label'] = score_label_df['worst_path_severity'].isin([0,1])

## Malignancy Prediction

In [ ]:
## Add Malignancy Predictions
predictions_wmal = ext.malignancy_for_models(score_label_df)

predictions_wmal.groupby('model_name').count()[['mal_pos_prob', 'mal_neg_prob', 'malignancy_pred']]

In [ ]:
__prob_sums = predictions_wmal[['mal_pos_prob', 'mal_neg_prob']].sum(axis=1).round(2).value_counts()
assert len(__prob_sums) == 1 and 1.0 in __prob_sums
__prob_sums

In [ ]:
for model in ext.models:
    predictions_wmal[predictions_wmal['model_name'] == model]['mal_pos_prob'].hist(alpha=0.5)

# Analysis
## Summary Statistics

In [ ]:
summary = ext.patient_summary_statistics(predictions_wmal)[['subgroup', 'value', 'count']]
summary.to_csv(ext.external_validation_path/f'{ext.environment}_summary.csv')
summary

## ROC

In [ ]:
ext.roc_figure_for_models(predictions_wmal)

# Export Key Results

## Margin ROC

In [ ]:
export_df = predictions_wmal.copy()

In [ ]:
margin_roc_df, margin_ci_df = ext.roc_dfs(export_df, 'margin', ext.margin_name_labels, 'margin_label')
margin_roc_df.head(5).pipe(display)
margin_ci_df.head(5).pipe(display)

In [ ]:
margin_roc_df.to_csv(ext.external_validation_path / f'{ext.environment}_margin_roc.csv', index=False)
margin_roc_df.groupby(['model', 'margin']).count().pipe(display)

margin_ci_df.to_csv(ext.external_validation_path / f'{ext.environment}_margin_ci.csv', index=False)
margin_ci_df[margin_ci_df['subgrouping'] == 'all'].pivot(index='margin', columns='model', values=['auc', 'ci_025', 'ci_975']).swaplevel(0,1,axis=1)[ext.models].round(2)

# Note the warnings come from the fact some subgroups only have one entry in the class. Obviously, that is not a valid test, but it's included for completeness.

## Activation Precision

In [ ]:
EXPORT_BOOTSTRAPS=100

In [ ]:
prototype_ap_df = ext.classwise_activation_precision_for_protopnets(num_bootstraps=EXPORT_BOOTSTRAPS)
prototype_ap_df

In [ ]:
gradcam_ap_df = ext.run_gradcams(num_bootstraps=EXPORT_BOOTSTRAPS)
gradcam_ap_df

In [ ]:
ap_ci_df = pd.concat([prototype_ap_df, gradcam_ap_df], sort=True)
ap_ci_df.to_csv(ext.external_validation_path / f'{ext.environment}_ap_ci.csv', index=False)
ap_ci_df.fillna('').groupby(['model', 'method']).mean().pipe(display)
ap_ci_df.pipe(display)

## Malignancy

In [ ]:
export_df[['mal_pos_prob', 'malignancy_label', 'model_name']].to_csv(ext.external_validation_path/f'{ext.environment}_mal_dist.csv', index=False)

In [ ]:
mal_roc_df, mal_ci_df = ext.roc_dfs(export_df, 'malignancy', ext.malignancy_name_labels, 'malignancy_label')
mal_roc_df.pipe(display)
mal_ci_df.pipe(display)

In [ ]:
mal_roc_df.to_csv(ext.external_validation_path / f'{ext.environment}_mal_roc.csv', index=False)
mal_roc_df.groupby(['model', 'malignancy']).count().pipe(display)

mal_ci_df.to_csv(ext.external_validation_path / f'{ext.environment}_mal_ci.csv', index=False)
mal_ci_df[mal_ci_df['subgrouping'] == 'all']\
    .pivot(index='malignancy', columns='model', values=['auc', 'ci_025', 'ci_975'])\
    .swaplevel(0,1,axis=1)[ext.models].round(2)

# Note the warnings come from the fact some subgroups only have one entry in the class. Obviously, that is not a valid test, but it's included for completeness.

In [ ]:
plt.figure()

# Plot ROC curve for each class
for model in ext.models:
    class_data = mal_roc_df[(mal_roc_df['model'] == model) & (mal_roc_df['malignancy'] == 'malignant')].sort_values('threshold')

    row = mal_ci_df[(mal_ci_df['subgrouping'] == 'all') & (mal_ci_df['model'] == model) & (mal_ci_df['malignancy'] == 'malignant')].round(2)

    auc_label = f'AUC: {row["auc"].item()} ({row["ci_025"].item()}, {row["ci_975"].item()})'

    plt.plot(class_data['fpr'], class_data['tpr'], label=f'Model {model}\n{auc_label}')

plt.plot([0, 1], [0, 1],'--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves for Malignancy')
plt.legend(loc='best')

# Show the plot
plt.show()